# 01 SAJU ML Data Integrity Audit V4

**Notebook version:** `SAJU_ML_DATA_INTEGRITY_AUDIT_V4_R2_20260815`

## Why this notebook exists

The frozen Ten-God architecture failed its predeclared NEW_DEV_2 external replication gate.
Before changing labels, features, models, or collecting another holdout, we must establish whether the
development evidence itself is trustworthy.

This notebook audits only **already-consumed development material**:

- `SAJU_ML_CORPUS_V1` development corpus
- `V2 NEW_DEV` only
- `NEW_DEV_2`, now treated as consumed diagnostic data

It **must not load**:

- V2 `NEW_CONFIRM`
- Public Life `CHECK`
- Public Life `FINAL`
- Validation B

## Audit dimensions

### A. Entity / identity integrity
- duplicate subject IDs
- duplicate normalized names
- cross-pool duplicate identity candidates
- same name with conflicting birth dates
- events occurring before birth
- implausibly young Career events
- event after an explicit death event

### B. Birth integrity
- exact birth time availability
- date/time parsing
- longitude/latitude validity
- UTC-offset validity
- source provenance
- Rodden rating where available

### C. Event integrity
- duplicate events
- same-subject/year conflicting Career labels
- missing event source
- single-source / second-source-needed status
- source-valence versus Career-label conflict
- low/medium target-assignment confidence
- event ownership / identity-risk heuristics

### D. Career target integrity
- Career pairability
- label counts
- event-type imbalance by valence
- transition events that should not automatically be negative
- Health-caused Career disruptions
- chronology/age confounding

### E. Chronology confounding
Pairwise Career models can accidentally exploit age structure if positive events systematically occur earlier
than negative events. Therefore we explicitly calculate:

- fraction of positive-negative pairs where positive occurred earlier
- positive and negative event age distributions
- an `AGE_ONLY` baseline where younger years are scored higher
- a `CALENDAR_YEAR_ONLY` baseline where earlier calendar years are scored higher

These are diagnostics, not model candidates.

## Important interpretation rule

This notebook does **not repair** records automatically.
It creates an issue ledger. Any repair after this audit changes the development corpus version and must be
traceable.

## R2 audit-logic correction

The original audit incorrectly treated every generic `event_type="death"` as the subject's own death.
R2 removes that inference. Only explicitly-owned subject-death event types can anchor an
`EVENT_AFTER_EXPLICIT_SUBJECT_DEATH` critical check. Generic death events are downgraded to
`DEATH_EVENT_OWNER_UNVERIFIED` and require entity/ownership review.

This correction changes the audit logic only. It does not mutate any corpus record or label.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import hashlib
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_DATA_INTEGRITY_AUDIT_V4_R2_20260815"

CAREER_DOMAIN = "Career"
VALID_TARGETS = {"positive", "negative", "unknown"}

# These are flags, not automatic exclusions.
CAREER_YOUNG_AGE_WARN = 16
CAREER_YOUNG_AGE_CRITICAL = 12
CAREER_MAX_AGE_WARN = 90

TRANSITION_TYPES = {
    "career_end", "retirement", "contract_end", "resignation",
    "career_transition", "band_breakup", "career_interruption",
}

HEALTH_MECHANISM_TYPES = {
    "career_injury", "career_accident", "illness", "injury",
    "health_crisis", "medical_leave",
}

NEGATIVE_OUTCOME_TYPES = {
    "fired", "election_loss", "major_loss", "show_cancellation",
    "commercial_failure", "critical_failure", "sanction",
    "qualification_failure", "leadership_loss",
}

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find repo root containing saju_engine.py. "
        "Run this notebook from inside the Chartpalja saju repository."
    )

REPO_ROOT = find_repo_root()

V1_PATH = REPO_ROOT / "research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json"
V2_DEV_PATH = REPO_ROOT / "research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json"
NEW_DEV_2_PATH = REPO_ROOT / "research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json"

OUT = REPO_ROOT / "research/ml/artifacts/v4_data_integrity_audit"
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 88)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)
print("repo:", REPO_ROOT)
print("output:", OUT)
print("=" * 88)

NOTEBOOK VERSION: SAJU_ML_DATA_INTEGRITY_AUDIT_V4_R2_20260815
repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_data_integrity_audit


## 1. Holdout-safe preflight

The audit refuses to fall back to the combined V2 file because doing so would load sealed `NEW_CONFIRM`.
A dedicated `SAJU_ML_CORPUS_V2_NEW_DEV.json` is required.

In [2]:
required = {
    "V1 development corpus": V1_PATH,
    "V2 NEW_DEV-only corpus": V2_DEV_PATH,
    "NEW_DEV_2 consumed corpus": NEW_DEV_2_PATH,
}

for label, path in required.items():
    ok = path.exists()
    print("%-34s %s  %s" % (label, "OK" if ok else "MISSING", path))
    if not ok:
        raise FileNotFoundError(
            "%s is required. The notebook will NOT fall back to a combined/sealed corpus."
            % path
        )

# Strong string-level guard: none of the configured input paths may be a sealed set.
configured_paths = [str(x) for x in required.values()]
for value in configured_paths:
    upper = value.upper()
    assert "NEW_CONFIRM" not in upper
    assert "FINAL" not in upper
    assert "CHECK" not in upper
    assert "VALIDATION_B" not in upper

print("\nHOLDOUT-SAFE INPUT PREFLIGHT: PASS")

V1 development corpus              OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json
V2 NEW_DEV-only corpus             OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json
NEW_DEV_2 consumed corpus          OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json

HOLDOUT-SAFE INPUT PREFLIGHT: PASS


## 2. Load only allowed development corpora

In [3]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

v1 = load_json(V1_PATH)
v2 = load_json(V2_DEV_PATH)
new2 = load_json(NEW_DEV_2_PATH)

datasets = {
    "V1": v1,
    "V2_NEW_DEV": v2,
    "NEW_DEV_2_CONSUMED": new2,
}

for pool, corpus in datasets.items():
    print(pool, "subjects =", len(corpus.get("subjects", [])))

# Defensive source-pool checks.
if "subjects" in v2:
    forbidden_v2 = [
        s.get("name")
        for s in v2["subjects"]
        if str(s.get("source_pool", "")).upper() == "NEW_CONFIRM"
    ]
    assert not forbidden_v2, "V2 NEW_DEV-only file contains NEW_CONFIRM subjects."

print("Allowed corpus load: PASS")

V1 subjects = 94
V2_NEW_DEV subjects = 70
NEW_DEV_2_CONSUMED subjects = 100
Allowed corpus load: PASS


## 3. Normalize subject and event tables

No labels are changed here.

In [4]:
def safe_date(value):
    if value is None:
        return None
    text = str(value).strip()
    try:
        return pd.Timestamp(text)
    except Exception:
        return None

def parse_offset_hours(raw):
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return np.nan
    if isinstance(raw, (int, float)):
        return float(raw)
    text = str(raw).strip()
    m = re.match(r"^([+-]?)(\d{1,2}):(\d{2})$", text)
    if not m:
        return np.nan
    sign = -1.0 if m.group(1) == "-" else 1.0
    return sign * (int(m.group(2)) + int(m.group(3)) / 60.0)

def normalize_name(name):
    if name is None:
        return ""
    x = str(name).casefold().strip()
    x = re.sub(r"\([^)]*\)", " ", x)
    x = re.sub(r"[^0-9a-z가-힣]+", "", x)
    return x

subject_rows = []
event_rows = []

for pool, corpus in datasets.items():
    for subject in corpus.get("subjects", []):
        birth = subject.get("birth") or {}
        birth_date = safe_date(birth.get("date"))
        birth_year = int(birth_date.year) if birth_date is not None else np.nan

        source_ref = (
            birth.get("source_ref")
            or birth.get("source_urls")
            or birth.get("source_url")
        )

        subject_rows.append({
            "pool": pool,
            "subject_id": subject.get("subject_id"),
            "name": subject.get("name"),
            "name_norm": normalize_name(subject.get("name")),
            "gender": subject.get("gender"),
            "occupation_category": subject.get("occupation_category"),
            "source_pool": subject.get("source_pool"),
            "origin_group": subject.get("origin_group"),
            "birth_date": birth.get("date"),
            "birth_year": birth_year,
            "birth_time": birth.get("time"),
            "birth_time_known": bool(birth.get("birth_time_known", False)),
            "birth_place": birth.get("place"),
            "timezone": birth.get("timezone"),
            "utc_offset": birth.get("utc_offset"),
            "utc_offset_hours": parse_offset_hours(birth.get("utc_offset")),
            "latitude": birth.get("latitude"),
            "longitude": birth.get("longitude"),
            "rodden_rating": birth.get("rodden_rating"),
            "birth_source_quality": birth.get("source_quality"),
            "birth_source_ref": source_ref,
            "birth_source_note": birth.get("source_note"),
        })

        for event in subject.get("events", []):
            targets = event.get("targets") or {}
            career = targets.get(CAREER_DOMAIN, "unknown")

            event_rows.append({
                "pool": pool,
                "subject_id": subject.get("subject_id"),
                "name": subject.get("name"),
                "name_norm": normalize_name(subject.get("name")),
                "birth_year": birth_year,
                "birth_time_known": bool(birth.get("birth_time_known", False)),
                "event_id": event.get("event_id"),
                "year": event.get("year"),
                "description": event.get("description"),
                "event_type": event.get("event_type"),
                "career_label": career,
                "source_valence": event.get("source_valence"),
                "target_assignment_confidence": event.get("target_assignment_confidence"),
                "target_assignment_method": event.get("target_assignment_method"),
                "source_event_confidence": event.get("source_event_confidence"),
                "source_weight": event.get("source_weight"),
                "source_ref": event.get("source_ref"),
                "source_quality": event.get("source_quality"),
                "source_verification_status": event.get("source_verification_status"),
                "needs_secondary_verification": event.get("needs_secondary_verification"),
                "ml_usable": bool(event.get("ml_usable", True)),
                "targets_json": json.dumps(targets, ensure_ascii=False, sort_keys=True),
            })

subjects = pd.DataFrame(subject_rows)
events = pd.DataFrame(event_rows)

events["year_numeric"] = pd.to_numeric(events["year"], errors="coerce")
events["event_age"] = events["year_numeric"] - events["birth_year"]

print("subjects:", len(subjects))
print("events:", len(events))
display(subjects.groupby("pool").size().rename("n_subjects"))
display(events.groupby("pool").size().rename("n_events"))

subjects: 264
events: 1152


pool
NEW_DEV_2_CONSUMED    100
V1                     94
V2_NEW_DEV             70
Name: n_subjects, dtype: int64

pool
NEW_DEV_2_CONSUMED    200
V1                    736
V2_NEW_DEV            216
Name: n_events, dtype: int64

## 4. Build issue ledger

Severity meaning:

- `CRITICAL`: record should not enter a gold training/evaluation set without correction
- `HIGH`: likely contamination / identity / chronology / label-definition problem
- `MEDIUM`: provenance or taxonomy weakness that must be reviewed
- `LOW`: descriptive QA / cleanup issue

In [5]:
issues = []

def add_issue(pool, subject_id, name, issue_code, severity, details, event_id=None, year=None):
    issues.append({
        "pool": pool,
        "subject_id": subject_id,
        "name": name,
        "event_id": event_id,
        "year": year,
        "issue_code": issue_code,
        "severity": severity,
        "details": details,
    })

# Subject-ID duplicates inside each corpus.
for (pool, sid), g in subjects.groupby(["pool", "subject_id"], dropna=False):
    if len(g) > 1:
        add_issue(pool, sid, g["name"].iloc[0], "DUPLICATE_SUBJECT_ID", "CRITICAL",
                  "subject_id occurs %d times in the same corpus" % len(g))

# Name duplicates inside a corpus.
for (pool, name_norm), g in subjects.groupby(["pool", "name_norm"], dropna=False):
    if name_norm and len(g) > 1:
        add_issue(pool, ",".join(g["subject_id"].astype(str)), " / ".join(g["name"].astype(str)),
                  "DUPLICATE_NORMALIZED_NAME_WITHIN_POOL", "HIGH",
                  "Normalized identity occurs %d times in pool" % len(g))

# Cross-pool identity candidates by normalized name.
for name_norm, g in subjects.groupby("name_norm"):
    if name_norm and g["pool"].nunique() > 1:
        add_issue(
            "CROSS_POOL",
            ",".join(g["subject_id"].astype(str)),
            " / ".join(g["name"].astype(str)),
            "CROSS_POOL_NAME_OVERLAP",
            "CRITICAL",
            "Same normalized name appears across pools: %s" % sorted(g["pool"].unique().tolist()),
        )

# Cross-pool same birth date + normalized name conflict detection.
identity_key = subjects["name_norm"].astype(str) + "|" + subjects["birth_date"].astype(str)
subjects["_identity_key"] = identity_key
for key, g in subjects.groupby("_identity_key"):
    if "|" in key and not key.startswith("|") and g["pool"].nunique() > 1:
        add_issue(
            "CROSS_POOL",
            ",".join(g["subject_id"].astype(str)),
            " / ".join(g["name"].astype(str)),
            "CROSS_POOL_NAME_BIRTH_MATCH",
            "CRITICAL",
            "Name+birth-date identity appears in multiple development pools.",
        )

# Birth checks.
for _, r in subjects.iterrows():
    if pd.isna(r["birth_year"]):
        add_issue(r["pool"], r["subject_id"], r["name"], "INVALID_BIRTH_DATE", "CRITICAL",
                  "Birth date could not be parsed.")

    if not r["birth_time_known"]:
        add_issue(r["pool"], r["subject_id"], r["name"], "BIRTH_TIME_UNKNOWN", "MEDIUM",
                  "Exact birth time unavailable; not eligible for strict-time modeling.")

    if r["birth_time_known"] and (r["birth_time"] is None or str(r["birth_time"]).strip() == ""):
        add_issue(r["pool"], r["subject_id"], r["name"], "BIRTH_TIME_FLAG_CONTRADICTION", "HIGH",
                  "birth_time_known=True but time is blank.")

    lat = pd.to_numeric(pd.Series([r["latitude"]]), errors="coerce").iloc[0]
    lon = pd.to_numeric(pd.Series([r["longitude"]]), errors="coerce").iloc[0]

    if not pd.isna(lat) and not (-90 <= lat <= 90):
        add_issue(r["pool"], r["subject_id"], r["name"], "INVALID_LATITUDE", "CRITICAL",
                  "Latitude outside [-90,90].")
    if not pd.isna(lon) and not (-180 <= lon <= 180):
        add_issue(r["pool"], r["subject_id"], r["name"], "INVALID_LONGITUDE", "CRITICAL",
                  "Longitude outside [-180,180].")

    if r["utc_offset"] is not None and pd.isna(r["utc_offset_hours"]):
        add_issue(r["pool"], r["subject_id"], r["name"], "UNPARSEABLE_UTC_OFFSET", "HIGH",
                  "UTC offset is present but could not be parsed.")

    if not r["birth_source_quality"]:
        add_issue(r["pool"], r["subject_id"], r["name"], "MISSING_BIRTH_SOURCE_QUALITY", "MEDIUM",
                  "Birth source quality is not recorded.")

# Event checks.
for _, r in events.iterrows():
    if r["career_label"] not in VALID_TARGETS:
        add_issue(r["pool"], r["subject_id"], r["name"], "INVALID_CAREER_LABEL", "CRITICAL",
                  "Unexpected Career target value: %r" % r["career_label"],
                  r["event_id"], r["year"])

    if pd.isna(r["year_numeric"]):
        add_issue(r["pool"], r["subject_id"], r["name"], "INVALID_EVENT_YEAR", "CRITICAL",
                  "Event year is not numeric.", r["event_id"], r["year"])
        continue

    if not pd.isna(r["birth_year"]) and r["year_numeric"] < r["birth_year"]:
        add_issue(r["pool"], r["subject_id"], r["name"], "EVENT_BEFORE_BIRTH", "CRITICAL",
                  "Event year %d precedes birth year %d." % (r["year_numeric"], r["birth_year"]),
                  r["event_id"], r["year"])

    if r["career_label"] in {"positive", "negative"}:
        if not pd.isna(r["event_age"]) and r["event_age"] < CAREER_YOUNG_AGE_CRITICAL:
            add_issue(r["pool"], r["subject_id"], r["name"], "CAREER_EVENT_IMPLAUSIBLY_YOUNG", "CRITICAL",
                      "Career event age %.1f < %d; strong identity-mismatch risk."
                      % (r["event_age"], CAREER_YOUNG_AGE_CRITICAL),
                      r["event_id"], r["year"])
        elif not pd.isna(r["event_age"]) and r["event_age"] < CAREER_YOUNG_AGE_WARN:
            add_issue(r["pool"], r["subject_id"], r["name"], "CAREER_EVENT_VERY_YOUNG", "HIGH",
                      "Career event age %.1f < %d; manual identity review required."
                      % (r["event_age"], CAREER_YOUNG_AGE_WARN),
                      r["event_id"], r["year"])

        if not pd.isna(r["event_age"]) and r["event_age"] > CAREER_MAX_AGE_WARN:
            add_issue(r["pool"], r["subject_id"], r["name"], "CAREER_EVENT_EXTREME_AGE", "MEDIUM",
                      "Career event age %.1f > %d; verify event ownership."
                      % (r["event_age"], CAREER_MAX_AGE_WARN),
                      r["event_id"], r["year"])

        if not r["source_ref"]:
            add_issue(r["pool"], r["subject_id"], r["name"], "CAREER_EVENT_MISSING_SOURCE", "HIGH",
                      "ML-usable Career label has no source_ref.",
                      r["event_id"], r["year"])

        if r["needs_secondary_verification"] is True:
            add_issue(r["pool"], r["subject_id"], r["name"], "SECOND_SOURCE_AUDIT_REQUIRED", "MEDIUM",
                      "Event explicitly requires secondary verification.",
                      r["event_id"], r["year"])

        if str(r["source_verification_status"]).upper() == "CURATED_NEEDS_SECOND_SOURCE_QA":
            add_issue(r["pool"], r["subject_id"], r["name"], "SECOND_SOURCE_AUDIT_REQUIRED", "MEDIUM",
                      "Legacy verification status requests a second source.",
                      r["event_id"], r["year"])

        if str(r["target_assignment_confidence"]).lower() in {"low", "medium"}:
            add_issue(r["pool"], r["subject_id"], r["name"], "NON_HIGH_TARGET_ASSIGNMENT", "MEDIUM",
                      "Career target assignment confidence is %s."
                      % r["target_assignment_confidence"],
                      r["event_id"], r["year"])

        source_valence = str(r["source_valence"]).lower()
        career_label = str(r["career_label"]).lower()
        if source_valence in {"positive", "negative"} and source_valence != career_label:
            add_issue(r["pool"], r["subject_id"], r["name"], "SOURCE_VALENCE_TARGET_CONFLICT", "HIGH",
                      "source_valence=%s but Career=%s" % (source_valence, career_label),
                      r["event_id"], r["year"])

        et = str(r["event_type"]).lower()
        if et in TRANSITION_TYPES:
            add_issue(r["pool"], r["subject_id"], r["name"], "CAREER_TRANSITION_TAXONOMY_REVIEW", "MEDIUM",
                      "Transition/end/resignation event should not be automatically assumed negative.",
                      r["event_id"], r["year"])

        if et in HEALTH_MECHANISM_TYPES:
            add_issue(r["pool"], r["subject_id"], r["name"], "CAREER_HEALTH_MECHANISM_REVIEW", "MEDIUM",
                      "Career outcome is caused by Health/accident mechanism; candidate multi-label event.",
                      r["event_id"], r["year"])

# Duplicate event IDs.
for (pool, eid), g in events[events["event_id"].notna()].groupby(["pool", "event_id"]):
    if len(g) > 1:
        add_issue(pool, g["subject_id"].iloc[0], g["name"].iloc[0],
                  "DUPLICATE_EVENT_ID", "CRITICAL",
                  "event_id occurs %d times." % len(g), eid, g["year"].iloc[0])

# Duplicate same subject-year-description.
dup_key_cols = ["pool", "subject_id", "year_numeric", "description"]
for key, g in events.groupby(dup_key_cols, dropna=False):
    if len(g) > 1:
        add_issue(g["pool"].iloc[0], g["subject_id"].iloc[0], g["name"].iloc[0],
                  "DUPLICATE_EVENT_CONTENT", "HIGH",
                  "Same subject/year/description occurs %d times." % len(g),
                  ",".join(g["event_id"].astype(str)), g["year"].iloc[0])

# Same-subject/year Career label collision.
career_labeled = events[
    events["ml_usable"]
    & events["career_label"].isin(["positive", "negative"])
].copy()

for (pool, sid, year), g in career_labeled.groupby(["pool", "subject_id", "year_numeric"]):
    if g["career_label"].nunique() > 1:
        add_issue(pool, sid, g["name"].iloc[0], "CAREER_SAME_YEAR_LABEL_COLLISION", "CRITICAL",
                  "Positive and negative Career labels occur in the same subject-year.",
                  ",".join(g["event_id"].astype(str)), year)

# Subject-death chronology audit.
#
# IMPORTANT R2 FIX:
# A generic event_type="death" does NOT prove that the subject died.
# In legacy corpora it may describe the death of a parent, spouse, colleague,
# political figure, etc. Treating every generic "death" as subject death
# produced false CRITICAL flags such as Arnold Schwarzenegger "dying" in 1972.
#
# We therefore hard-fail only when the schema explicitly identifies the
# event as the subject's own death. Generic death-like events are surfaced
# for manual ownership review instead of being used as a chronology anchor.

explicit_subject_death_types = {
    "subject_death",
    "death_of_subject",
    "self_death",
    "fatal_accident_subject",
}

generic_death_types = {
    "death",
    "career_accident_death",
    "fatal_accident",
}

events["_event_type_norm"] = (
    events["event_type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

for _, r in events[
    events["_event_type_norm"].isin(generic_death_types)
].iterrows():
    add_issue(
        r["pool"],
        r["subject_id"],
        r["name"],
        "DEATH_EVENT_OWNER_UNVERIFIED",
        "MEDIUM",
        "Death-like event exists, but the current schema does not prove "
        "that the deceased person is the subject. Manual/entity-ID review required.",
        r["event_id"],
        r["year"],
    )

subject_death_mask = events["_event_type_norm"].isin(
    explicit_subject_death_types
)

for (pool, sid), g in events.groupby(["pool", "subject_id"]):
    subject_death_years = g.loc[
        subject_death_mask.loc[g.index],
        "year_numeric",
    ].dropna().tolist()

    if not subject_death_years:
        continue

    subject_death_year = min(subject_death_years)

    later = g[
        g["year_numeric"] > subject_death_year
    ]

    for _, r in later.iterrows():
        add_issue(
            pool,
            sid,
            r["name"],
            "EVENT_AFTER_EXPLICIT_SUBJECT_DEATH",
            "CRITICAL",
            "Event occurs after explicitly-owned subject death year %d."
            % subject_death_year,
            r["event_id"],
            r["year"],
        )

issue_df = pd.DataFrame(issues)
if len(issue_df):
    severity_order = pd.Categorical(
        issue_df["severity"],
        categories=["CRITICAL", "HIGH", "MEDIUM", "LOW"],
        ordered=True,
    )
    issue_df = issue_df.assign(_sev=severity_order).sort_values(
        ["_sev", "pool", "subject_id", "year"]
    ).drop(columns=["_sev"])

issue_path = OUT / "V4_DATA_INTEGRITY_ISSUE_LEDGER.csv"
issue_df.to_csv(issue_path, index=False)

print("issues:", len(issue_df))
if len(issue_df):
    display(
        issue_df.groupby(["severity", "issue_code"])
        .size()
        .reset_index(name="n")
        .sort_values(["severity", "n"], ascending=[True, False])
    )

issues: 1071


,severity,issue_code,n
1,CRITICAL,CAREER_SAME_YEAR_LABEL_COLLISION,5
0,CRITICAL,CAREER_EVENT_IMPLAUSIBLY_YOUNG,2
2,HIGH,CAREER_EVENT_MISSING_SOURCE,227
3,HIGH,CAREER_EVENT_VERY_YOUNG,2
9,MEDIUM,SECOND_SOURCE_AUDIT_REQUIRED,414
8,MEDIUM,NON_HIGH_TARGET_ASSIGNMENT,317
7,MEDIUM,DEATH_EVENT_OWNER_UNVERIFIED,38
4,MEDIUM,BIRTH_TIME_UNKNOWN,27
6,MEDIUM,CAREER_TRANSITION_TAXONOMY_REVIEW,20
5,MEDIUM,CAREER_HEALTH_MECHANISM_REVIEW,19


## 5. Career pairability and chronology confounding

This section is critical.

A pair is `(positive Career year, negative Career year)` within the same person.
If positive events systematically occur earlier in life, a model containing age-dependent structures such as
Daewoon may look predictive even when it is partially exploiting life-cycle ordering.

In [6]:
career = career_labeled.copy()

# Collapse identical-direction duplicate labels in the same subject-year.
collapsed_rows = []

for (pool, sid, year), g in career.groupby(["pool", "subject_id", "year_numeric"]):
    if g["career_label"].nunique() != 1:
        continue
    collapsed_rows.append({
        "pool": pool,
        "subject_id": sid,
        "name": g["name"].iloc[0],
        "birth_year": g["birth_year"].iloc[0],
        "year": int(year),
        "label": g["career_label"].iloc[0],
        "age": float(g["event_age"].iloc[0]) if not pd.isna(g["event_age"].iloc[0]) else np.nan,
        "confidence": g["source_event_confidence"].iloc[0],
    })

career_collapsed = pd.DataFrame(collapsed_rows)

pair_rows = []
subject_pairability_rows = []

for (pool, sid), g in career_collapsed.groupby(["pool", "subject_id"]):
    pos = g[g["label"] == "positive"]
    neg = g[g["label"] == "negative"]
    pairable = len(pos) > 0 and len(neg) > 0

    subject_pairability_rows.append({
        "pool": pool,
        "subject_id": sid,
        "name": g["name"].iloc[0],
        "n_positive_years": len(pos),
        "n_negative_years": len(neg),
        "pairable": pairable,
        "n_pairs": int(len(pos) * len(neg)),
    })

    if not pairable:
        continue

    for _, p in pos.iterrows():
        for _, n in neg.iterrows():
            age_delta = p["age"] - n["age"] if not (pd.isna(p["age"]) or pd.isna(n["age"])) else np.nan
            year_delta = p["year"] - n["year"]

            pair_rows.append({
                "pool": pool,
                "subject_id": sid,
                "name": p["name"],
                "positive_year": int(p["year"]),
                "negative_year": int(n["year"]),
                "positive_age": p["age"],
                "negative_age": n["age"],
                "positive_minus_negative_age": age_delta,
                "positive_minus_negative_year": year_delta,
                "positive_earlier": bool(p["year"] < n["year"]),
                "same_year": bool(p["year"] == n["year"]),
                # Baseline score = -age means younger years rank higher.
                "age_only_pairwise": (
                    1.0 if p["age"] < n["age"]
                    else 0.5 if p["age"] == n["age"]
                    else 0.0
                ) if not (pd.isna(p["age"]) or pd.isna(n["age"])) else np.nan,
                # Equivalent calendar-time diagnostic.
                "earlier_year_pairwise": (
                    1.0 if p["year"] < n["year"]
                    else 0.5 if p["year"] == n["year"]
                    else 0.0
                ),
            })

pairability_df = pd.DataFrame(subject_pairability_rows)
career_pairs = pd.DataFrame(pair_rows)

pairability_df.to_csv(OUT / "V4_CAREER_SUBJECT_PAIRABILITY.csv", index=False)
career_pairs.to_csv(OUT / "V4_CAREER_PAIR_CHRONOLOGY.csv", index=False)

chronology_rows = []

for pool, g in career_pairs.groupby("pool"):
    chronology_rows.append({
        "pool": pool,
        "n_subjects": g["subject_id"].nunique(),
        "n_pairs": len(g),
        "positive_earlier_share": float(g["positive_earlier"].mean()),
        "mean_positive_age": float(g["positive_age"].mean()),
        "mean_negative_age": float(g["negative_age"].mean()),
        "median_positive_minus_negative_age": float(g["positive_minus_negative_age"].median()),
        "AGE_ONLY_pairwise": float(g["age_only_pairwise"].mean()),
        "EARLIER_YEAR_ONLY_pairwise": float(g["earlier_year_pairwise"].mean()),
    })

chronology_summary = pd.DataFrame(chronology_rows)
chronology_summary.to_csv(OUT / "V4_CAREER_CHRONOLOGY_SUMMARY.csv", index=False)

display(
    pairability_df.groupby("pool").agg(
        subjects=("subject_id", "nunique"),
        pairable_subjects=("pairable", "sum"),
        total_pairs=("n_pairs", "sum"),
    ).reset_index()
)

print("\nChronology confounding diagnostics:")
display(chronology_summary)

,pool,subjects,pairable_subjects,total_pairs
0,NEW_DEV_2_CONSUMED,100,100,100
1,V1,88,34,196
2,V2_NEW_DEV,70,70,152



Chronology confounding diagnostics:


,pool,n_subjects,n_pairs,positive_earlier_share,mean_positive_age,mean_negative_age,median_positive_minus_negative_age,AGE_ONLY_pairwise,EARLIER_YEAR_ONLY_pairwise
0,NEW_DEV_2_CONSUMED,100,100,0.880000,33.730000,41.000000,-6.0,0.880000,0.880000
1,V1,34,196,0.632653,45.188776,49.387755,-3.5,0.632653,0.632653
2,V2_NEW_DEV,70,152,0.763158,39.006579,47.026316,-7.0,0.763158,0.763158


## 6. Event-type / valence structure

If certain event mechanisms almost always occur only as positive or only as negative, `Career` may be mixing
different target phenomena instead of describing one coherent outcome.

In [7]:
career_event_types = (
    career
    .groupby(["pool", "event_type", "career_label"])
    .size()
    .reset_index(name="n")
)

career_event_type_pivot = (
    career_event_types
    .pivot_table(
        index=["pool", "event_type"],
        columns="career_label",
        values="n",
        fill_value=0,
    )
    .reset_index()
)

career_event_type_pivot.to_csv(
    OUT / "V4_CAREER_EVENT_TYPE_BY_VALENCE.csv",
    index=False,
)

display(
    career_event_type_pivot
    .sort_values(
        ["pool"] + [c for c in ["negative", "positive"] if c in career_event_type_pivot.columns],
        ascending=False,
    )
    .head(80)
)

career_label,pool,event_type,negative,positive
43,V2_NEW_DEV,career_loss,74,0
45,V2_NEW_DEV,scandal,2,0
44,V2_NEW_DEV,career_success,0,138
42,V2_NEW_DEV,award,0,2
40,V1,other,17,32
36,V1,election,11,47
33,V1,career_loss,10,0
32,V1,career,7,36
39,V1,legal,7,2
38,V1,illness,3,0


## 7. Source / confidence summary

In [8]:
source_summary = (
    career
    .groupby("pool")
    .agg(
        n_career_events=("event_id", "size"),
        source_missing_share=("source_ref", lambda x: float(x.isna().mean())),
        high_event_conf_share=(
            "source_event_confidence",
            lambda x: float((x.astype(str).str.lower() == "high").mean()),
        ),
        high_assignment_conf_share=(
            "target_assignment_confidence",
            lambda x: float((x.astype(str).str.lower() == "high").mean()),
        ),
        second_source_flag_share=(
            "needs_secondary_verification",
            lambda x: float((x == True).mean()),
        ),
    )
    .reset_index()
)

source_summary.to_csv(
    OUT / "V4_CAREER_SOURCE_QUALITY_SUMMARY.csv",
    index=False,
)

display(source_summary)

,pool,n_career_events,source_missing_share,high_event_conf_share,high_assignment_conf_share,second_source_flag_share
0,NEW_DEV_2_CONSUMED,200,0.000000,0.815000,0.815000,0.99
1,V1,315,0.720635,0.812698,0.333333,0.00
2,V2_NEW_DEV,216,0.000000,0.675926,0.675926,0.00


## 8. Subject-level gold eligibility

This does not mutate the corpus.

`gold_candidate=False` if a subject has any `CRITICAL` or `HIGH` issue.
`strict_time_candidate` additionally requires exact birth time.

In [9]:
subject_issue_severity = (
    issue_df[
        issue_df["pool"].isin(subjects["pool"].unique())
    ]
    .groupby(["pool", "subject_id", "severity"])
    .size()
    .unstack(fill_value=0)
    if len(issue_df)
    else pd.DataFrame()
)

gold = subjects[
    [
        "pool", "subject_id", "name", "birth_time_known",
        "birth_date", "birth_time", "birth_source_quality",
        "rodden_rating",
    ]
].copy()

if len(subject_issue_severity):
    gold = gold.merge(
        subject_issue_severity.reset_index(),
        on=["pool", "subject_id"],
        how="left",
    )

for col in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    if col not in gold.columns:
        gold[col] = 0
    gold[col] = gold[col].fillna(0).astype(int)

gold = gold.merge(
    pairability_df[
        ["pool", "subject_id", "pairable", "n_pairs"]
    ],
    on=["pool", "subject_id"],
    how="left",
)

gold["pairable"] = gold["pairable"].fillna(False)
gold["n_pairs"] = gold["n_pairs"].fillna(0).astype(int)

gold["gold_candidate"] = (
    (gold["CRITICAL"] == 0)
    & (gold["HIGH"] == 0)
)

gold["strict_time_gold_candidate"] = (
    gold["gold_candidate"]
    & gold["birth_time_known"]
    & gold["pairable"]
)

gold.to_csv(
    OUT / "V4_SUBJECT_GOLD_ELIGIBILITY.csv",
    index=False,
)

display(
    gold.groupby("pool").agg(
        subjects=("subject_id", "nunique"),
        gold_candidate=("gold_candidate", "sum"),
        strict_time_gold_candidate=("strict_time_gold_candidate", "sum"),
        pairable=("pairable", "sum"),
        critical_subjects=("CRITICAL", lambda x: int((x > 0).sum())),
        high_issue_subjects=("HIGH", lambda x: int((x > 0).sum())),
    ).reset_index()
)

,pool,subjects,gold_candidate,strict_time_gold_candidate,pairable,critical_subjects,high_issue_subjects
0,NEW_DEV_2_CONSUMED,100,99,99,100,1,0
1,V1,94,42,11,34,5,52
2,V2_NEW_DEV,70,70,70,70,0,0


## 9. Audit decision

The audit intentionally distinguishes:

- **data defects**: identity/chronology/collisions
- **provenance weaknesses**: missing or single-source events
- **target-design risk**: chronology and heterogeneous event mechanisms

A corpus can be structurally valid yet still be unsuitable as a clean gold benchmark.

In [10]:
critical_count = int((issue_df["severity"] == "CRITICAL").sum()) if len(issue_df) else 0
high_count = int((issue_df["severity"] == "HIGH").sum()) if len(issue_df) else 0

chronology_risk_pools = []

for _, r in chronology_summary.iterrows():
    # Descriptive risk criterion, not significance testing.
    if (
        r["positive_earlier_share"] >= 0.70
        or r["AGE_ONLY_pairwise"] >= 0.65
        or r["EARLIER_YEAR_ONLY_pairwise"] >= 0.65
    ):
        chronology_risk_pools.append(r["pool"])

if critical_count > 0:
    status = "FAIL_CRITICAL_DATA_DEFECTS_PRESENT"
elif high_count > 0:
    status = "HOLD_HIGH_PRIORITY_DATA_REVIEW_REQUIRED"
elif chronology_risk_pools:
    status = "HOLD_TARGET_CHRONOLOGY_CONFOUNDING"
else:
    status = "PASS_STRUCTURAL_AUDIT_WITH_PROVENANCE_REVIEW"

decision = {
    "version": "SAJU_ML_DATA_INTEGRITY_AUDIT_DECISION_V4",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": status,
    "allowed_pools_only": list(datasets.keys()),
    "sealed_data_loaded": False,
    "issue_counts": {
        "critical": critical_count,
        "high": high_count,
        "medium": int((issue_df["severity"] == "MEDIUM").sum()) if len(issue_df) else 0,
        "low": int((issue_df["severity"] == "LOW").sum()) if len(issue_df) else 0,
    },
    "chronology_risk_pools": chronology_risk_pools,
    "rules": {
        "do_not_auto_repair": True,
        "do_not_open_new_confirm": True,
        "new_dev_2_remains_consumed": True,
        "next_step": (
            "Resolve critical/high identity and event defects, then version a cleaned DEV_V4. "
            "Do not run a new feature/model tournament until feature parity audit also passes."
        ),
    },
}

with open(
    OUT / "V4_DATA_INTEGRITY_AUDIT_DECISION.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(decision, f, ensure_ascii=False, indent=2)

print(json.dumps(decision, ensure_ascii=False, indent=2))

{
  "version": "SAJU_ML_DATA_INTEGRITY_AUDIT_DECISION_V4",
  "notebook_version": "SAJU_ML_DATA_INTEGRITY_AUDIT_V4_R2_20260815",
  "created_at": "2026-08-15T22:29:33",
  "status": "FAIL_CRITICAL_DATA_DEFECTS_PRESENT",
  "allowed_pools_only": [
    "V1",
    "V2_NEW_DEV",
    "NEW_DEV_2_CONSUMED"
  ],
  "sealed_data_loaded": false,
  "issue_counts": {
    "critical": 7,
    "high": 229,
    "medium": 835,
    "low": 0
  },
  "chronology_risk_pools": [
    "NEW_DEV_2_CONSUMED",
    "V2_NEW_DEV"
  ],
  "rules": {
    "do_not_auto_repair": true,
    "do_not_open_new_confirm": true,
    "new_dev_2_remains_consumed": true,
    "next_step": "Resolve critical/high identity and event defects, then version a cleaned DEV_V4. Do not run a new feature/model tournament until feature parity audit also passes."
  }
}


## Files to send back

After **Restart Kernel -> Run All**, send at minimum:

```text
research/ml/artifacts/v4_data_integrity_audit/
├── V4_DATA_INTEGRITY_AUDIT_DECISION.json
├── V4_DATA_INTEGRITY_ISSUE_LEDGER.csv
├── V4_SUBJECT_GOLD_ELIGIBILITY.csv
├── V4_CAREER_CHRONOLOGY_SUMMARY.csv
├── V4_CAREER_EVENT_TYPE_BY_VALENCE.csv
└── V4_CAREER_SOURCE_QUALITY_SUMMARY.csv
```

Do not repair labels inside this notebook.